In [2]:
from truveta.study import Client, OutputMode
import pyspark.pandas as ps
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

In [15]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
#client = Client(output_mode = OutputMode.PySpark)
study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
population = study.get_population(title = "Control Group")
#population = study.get_population(id = "p-y4vjjpkak2tebbzl646avkf5he")
# population
# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()
#snapshot

In [16]:
output_path_local = study.get_output_path(fs = True)
file_to_read = output_path_local + "/test_t3.csv"
test = pd.read_csv(file_to_read)
file_to_read = output_path_local + "/control_t1.csv"
control = pd.read_csv(file_to_read)
# save the med/ med_wo_t2d label
file_to_write = output_path_local + "/drug_source_label.csv"
drug_source_label = pd.read_csv(file_to_write)
file_to_write = output_path_local + "/drugexporsure.csv"
drugexporsure = pd.read_csv(file_to_write)
print(test.shape, control.shape, drug_source_label.shape, drugexporsure.shape)

In [17]:
file_to_write = output_path_local + "/test_zcodecount.csv"
test_zcodecount = pd.read_csv(file_to_write)
test = test.merge(test_zcodecount, on = "PersonId")
test.shape

In [18]:
med_full_df = test[test.source_type == 'med'].copy()
surgery_df = test[test.source_type == 'surgery'].copy()
print(len(med_full_df), len(surgery_df))
med_noexppreg = med_full_df.copy()
med_noexppreg = med_noexppreg.drop('source_type', axis=1)
med_noexppreg = med_noexppreg.merge(drug_source_label, on = 'PersonId', how = 'right')
med_noexppreg.source_type.value_counts()
start_drug_preg = med_full_df[~med_full_df.PersonId.isin(med_noexppreg.PersonId)].copy()
print(len(start_drug_preg))
test = test[~test.PersonId.isin(start_drug_preg.PersonId)].copy()
print(len(test))

In [19]:
test.source_type.value_counts()

In [20]:
test_df = test[test.source_type == 'med'].copy()
test_df['preg_related_htn'] = (
    (test['gest_hyper_no_prior_hyper'] == True) |
    (test['preeclampsia_no_prior_hyper'] == True)
)

control['preg_related_htn'] = (
    (control['gest_hyper_no_prior_hyper'] == True) |
    (control['preeclampsia_no_prior_hyper'] == True)
)

print(test_df['preg_related_htn'].value_counts(), control['preg_related_htn'].value_counts())

In [21]:
test_df.gest_hyper.value_counts(), control.gest_hyper.value_counts()

In [22]:
print(test_df.t2d_before_pregnancy.value_counts(), test_df.hyper_before_pregnancy.value_counts())
print(control.t2d_before_pregnancy.value_counts(), control.hyper_before_pregnancy.value_counts())

In [23]:
print(test_df.gest_diabetes_no_prior_t2d.value_counts(), test_df.preg_related_htn.value_counts())
print(control.gest_diabetes_no_prior_t2d.value_counts(), control.preg_related_htn.value_counts())

In [24]:
overlap = set(test_df['PersonId']).intersection(set(control['PersonId']))

print(overlap)
print(len(overlap))

### join the medication drug expo dataset - Limited details on the "pregnancy exposed" group

In [25]:
test_df = test_df.merge(drugexporsure, on = "PersonId", how = "left")
test_df.shape, test_df.ExposedInPregnancy.value_counts()

In [26]:
import matplotlib.pyplot as plt
mode_week = test_df["gestational_week"].round().mode()[0]
median_week = test_df["gestational_week"].round().median()
plt.figure()
test_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(mode_week, linestyle="--", label=f"Median week: {median_week}")

plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Medication Group Distribution of gestational age at delivery")

plt.legend()
plt.show()

##### into 3 groups

In [27]:
df = test_df.copy()

# Ensure datetime
df['estimated_LMP'] = pd.to_datetime(df['estimated_LMP'])
df['DrugStart'] = pd.to_datetime(df['DrugStart'])
df['DrugEnd'] = pd.to_datetime(df['DrugEnd'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

# Trimester boundaries (clinical definition)
df['T1_end'] = df['estimated_LMP'] + pd.Timedelta(weeks=13, days=6)
df['T2_start'] = df['estimated_LMP'] + pd.Timedelta(weeks=14)
df['T2_end'] = df['estimated_LMP'] + pd.Timedelta(weeks=27, days=6)
df['T3_start'] = df['estimated_LMP'] + pd.Timedelta(weeks=28)
df['T3_end'] = df['estimated_LMP'] + pd.Timedelta(weeks=40, days=6)

In [28]:
# Exposure flags based on overlap
df['exposed_T1'] = (
    (df['DrugStart'] <= df['T1_end']) &
    (df['DrugEnd'] >= df['estimated_LMP'])
)

df['exposed_T2'] = (
    (df['DrugStart'] <= df['T2_end']) &
    (df['DrugEnd'] >= df['T2_start'])
)

df['exposed_T3'] = (
    (df['DrugStart'] <= df['T3_end']) &
    (df['DrugEnd'] >= df['T3_start'])
)

In [29]:
def classify_exposure(row):
    t1, t2, t3 = row['exposed_T1'], row['exposed_T2'], row['exposed_T3']
    
    if t1 and not t2 and not t3:
        return 'T1_only'
    elif t1 and t2 and not t3:
        return 'T1_T2'
    elif t1 and t2 and t3:
        return 'Throughout'
    elif not t1 and t2 and not t3:
        return 'T2_only'
    elif not t1 and not t2 and t3:
        return 'T3_only'
    elif not t1 and t2 and t3:
        return 'T2_T3'   # optional but often useful
    else:
        return 'No_exposure'

df['pregnancy_exposure_group'] = df.apply(classify_exposure, axis=1)

In [30]:
df['user_group'] = df['pregnancy_exposure_group'].replace({
    'No_exposure': 'former_user',
    'T1_only': 'continued_user',
    'T1_T2': 'continued_user',
    'Throughout': 'continued_user'
})

##### checking new refill after pregnancy

In [31]:
df_preg = df[df['user_group'] == "continued_user"].copy()
df_preg['new_refill'] = df_preg['NumDrugEpisodes'] > 1
df_preg['new_refill'].value_counts()

##### add interval between last supply and pregnancy date - former user

In [32]:
df_former = df[df['user_group'] == 'former_user'].copy()

df_former['days_between_stop_and_preg'] = (
    df_former['estimated_LMP'] - df_former['DrugEnd']
).dt.days

df_former['days_between_stop_and_preg'].describe()

##### how much supply typical dispensed - 28 days of supply
in here the supply days is define as DrugEnd - DrugStart

In [39]:
df['supply_days'] = (
    pd.to_datetime(df['DrugEnd']) - pd.to_datetime(df['DrugStart'])
).dt.days

summary_supply = df.groupby('pregnancy_exposure_group').agg(
    mean_supply=('supply_days', 'mean'),
    std_supply=('supply_days', 'std'),
    median_supply=('supply_days', 'median')
)
summary_supply

In [40]:
file_to_write = output_path_local + "/drug_dayssupply.csv"
daysupp = pd.read_csv(file_to_write)
df['PersonId'] = df['PersonId'].astype(str)
daysupp['PersonId'] = daysupp['PersonId'].astype(str)

df = df.merge(daysupp[['PersonId', 'Supply']], on='PersonId', how='left')
df.Supply.describe()

In [41]:
df.columns

In [42]:
from tableone import TableOne

columns = ['Supply']
categorical = []
groupby = 'pregnancy_exposure_group'

table1 = TableOne(
    df,
    columns=columns,
    categorical=categorical,
    groupby=groupby,
    nonnormal=['Supply'],
    pval=True 
)

table1

In [43]:
control_df = control.copy()
print(control_df.shape)
control_df = control_df.dropna(subset=["PrePregnancyBMI"]).copy()
print(control_df.shape)
control_df['user_group'] = 'non_user'

In [24]:
import matplotlib.pyplot as plt
mode_week = control_df["gestational_week"].round().mode()[0]
median_week = control_df["gestational_week"].round().median()
plt.figure()
control_df["gestational_week"].hist(bins=40)
plt.axvline(mode_week, linestyle="--", label=f"Most common week: {mode_week}")
plt.axvline(mode_week, linestyle="--", label=f"Median week: {median_week}")

plt.xlabel("Gestational age (weeks)")
plt.ylabel("Count")
plt.title("Control Group Distribution of gestational age at delivery")

plt.legend()
plt.show()

### Updating Control and Test Dataset - Don't Run

In [26]:
'''
if wanted this can be apply in the beginning, but for test purpose, just keep it simple
'''
def load_condition_data(
    snapshot,
    codeset_url=None,
    code_set=None,
    codes = 'codes',
    table_name="Condition",
    view_name="tbl_index_condition",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId",
    verbose=True
):
    """
    parameter:
        snapshot: Truveta snapshot 
        codeset_url: if use prose URL load codeset, fill out this
        code_set: if use manual input code snapshot.codeset(...) use this
        table_name: 'Condition', 'Procedure'
        view_name: sql table
        concept_map_table: mapping 'ConditionCodeConceptMap'
        concept_map_key: mapping key 'CodeConceptMapId'
        verbose: print out?
    
    return:
        matched_df: after mathcing pandas DataFrame
        unique_person_count: counting number of PersonId
    """
    # support two ways from prose URL load the code_set
    if code_set is None:
        if codeset_url is None:
            raise ValueError("Must provide either `codeset_url` or `code_set`")
        code_set = snapshot.codeset_from_prose(url=codeset_url, variable_name=codes)

    # temp table
    index_table = snapshot.load_filtered_table(table_name, code_set, view_name=view_name)

    if verbose:
        print(f"[{table_name}] Unique PersonId (after code filter):", index_table['PersonId'].nunique())

    # SQL match code
    sql_query = f"""
        SELECT m.PersonId, m.RecordedDateTime, pm.* 
        FROM {view_name} m 
        JOIN {concept_map_table} pm ON m.{concept_map_key} = pm.Id
    """
    df = ps.sql(sql_query).to_pandas()

    # getting the code
    matched_df = match_code(df, code_set)

    if verbose:
        print("Total matched rows:", len(matched_df))
        print("Unique PersonId (after match):", matched_df['PersonId'].nunique())

    return matched_df, matched_df['PersonId'].nunique()


## Running the decode_concepts function - use for match the concept code with the actual name
from typing import overload
import pyspark.pandas as ps
import pandas as pd
from pyspark.sql import DataFrame

@overload
def decode_concepts(df: pd.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> pd.DataFrame: ...
@overload
def decode_concepts(df: ps.DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> ps.DataFrame: ...
@overload
def decode_concepts(df: DataFrame, drop_concepts: bool = True, columns: list[str] | None = None) -> DataFrame: ...
def decode_concepts(df: pd.DataFrame | ps.DataFrame | DataFrame, drop_concepts: bool = True, columns: list[str] | None = None):
    """
    decodes the top level *ConceptId columns within the given data frame and derives new names (without the ConceptId suffix). 
    It assume that every column ending with `ConceptId` which is an integer or float is a concept column
    :param df the data frame to decode
    :param drop_concepts whether to drop *ConceptId columns (default true)
    :param columns optional direct list of columns to decode disabling the auto infering
    :returns the enhanced data frame
    """
    def should_decode(col: str, dtype: str) -> bool:
        if columns:
            return col in columns
        return col.endswith('ConceptId') and str(dtype) in ('int', 'int32', 'float64', 'float32')
    
    column_names = set(df.columns)

    def target_col(col: str) -> str:
        name = col.removesuffix('ConceptId')
        if name == col and drop_concepts:
            # return the same name
            return name
        while name in column_names:
            name = f"{name}Name"
        return name
    
    def safe_name(name: str) -> str:
        while name in column_names:
            name = f"{name}_tmp"
        return name

    final_order: list[str] = []
    
    if isinstance(df, pd.DataFrame):
        lookup = None
        for col, dtype in zip(df.columns, df.dtypes):
            if should_decode(col, str(dtype)):
                name = target_col(col)
                column_names.add(name)
                if lookup is None:
                    # lazy lookup
                    lookup = spark.sql("SELECT ConceptId, ConceptName FROM Concept").toPandas().set_index("ConceptId").ConceptName
                df[name] = df[col].map(lookup)
                if drop_concepts and name != col:
                    df = df.drop(columns=[col])
                else:
                    final_order.append(col) # keep original
                final_order.append(name)
            else:
                final_order.append(col)
        return df[final_order]

    return_pandas = False
    if isinstance(df, ps.DataFrame):
        return_pandas = True
        df = df.to_spark()
    
    concepts_s = spark.sql("SELECT ConceptId, ConceptName FROM Concept").cache()
    for col, dtype in df.dtypes:
        if should_decode(col, str(dtype)):
            name = target_col(col)
            column_names.add(name)
            if col == name and drop_concepts:
                # need to write to the same name, rename old one
                tmp_name = safe_name(col)
                df = df.withColumnRenamed(col, tmp_name).join(concepts_s.withColumnRenamed("ConceptId", tmp_name).withColumnRenamed("ConceptName", name), on=tmp_name, how="left").drop(tmp_name)
            else:
                df = df.join(concepts_s.withColumnRenamed("ConceptId", col).withColumnRenamed("ConceptName", name), on=col, how="left")
                if drop_concepts:
                    df = df.drop(col)
                else:
                    final_order.append(col)
            final_order.append(name)
        else:
            final_order.append(col)
    df = df.select(final_order)
    return df.pandas_api() if return_pandas else df

def match_code(df, codes_df):
    code_name = decode_concepts(df)
    case_names = codes_df.ConceptName.to_pandas().tolist()
    code_name = code_name[code_name['Code'].isin(case_names)]
    return code_name

In [27]:
defPrenatal = "/definitions/prenatal"
Prenatal, count = load_condition_data(
    snapshot, 
    codeset_url=defPrenatal,
    table_name="Condition",
    codes="Prenatal",
    view_name="tbl_index_Prenatal",
    concept_map_table="ConditionCodeConceptMap",
    concept_map_key="CodeConceptMapId"
)
csection_code = snapshot.codeset_from_prose(url = "/definitions/c-section", variable_name = "codes")
index_c = snapshot.load_filtered_table("Procedure", csection_code, view_name = 'tbl_index_c')
print(index_c['PersonId'].nunique())
df = ps.sql("SELECT p.PersonId, p.RecordedDateTime, p.StartDateTime, pm.* FROM tbl_index_c p JOIN ProcedureCodeConceptMap pm on p.CodeConceptMapId = pm.Id").to_pandas()
csection = match_code(df, csection_code)
csection['StartDateTime'] = csection['StartDateTime'].fillna(csection['RecordedDateTime'])
csection = csection.drop(columns=['RecordedDateTime'])
csection = csection.rename(columns={'StartDateTime': 'RecordedDateTime'})
csection.head()

In [28]:
control_df = control_df.drop(columns=['obstetric_care', 'csection1'])

In [29]:
# check if those condition happens during pregnancy
def mark_condition_in_pregnancy(condition_df, delivery_df, col_name):
    """
    check if condition happen in preg, mark T/F to condition_col_name
    parameter:
        condition_df: condition record - table 3 variables
        delivery_df: PersonId, estimated_LMP, delivery_date, record for delivery/preg
        col_name
        
    return updated delivery_df with new col T/F
    """
    merged = condition_df.merge(
        delivery_df[['PersonId', 'estimated_LMP', 'delivery_date']],
        on='PersonId',
        how='left'
    )

    #check
    merged['in_pregnancy'] = (
        (merged['RecordedDateTime'] >= merged['estimated_LMP']) &
        (merged['RecordedDateTime'] <= merged['delivery_date'])
    )

    # keep satisfied PersonId
    flagged_ids = merged.loc[merged['in_pregnancy'], 'PersonId'].drop_duplicates()
    condition_flag = pd.DataFrame({ 'PersonId': flagged_ids, col_name: True })

    # merge back delivery_df others are False
    delivery_df = delivery_df.merge(condition_flag, on='PersonId', how='left')
    delivery_df[col_name] = delivery_df[col_name].fillna(False)

    return delivery_df

In [30]:
prenatal_ids = set(Prenatal['PersonId'].dropna().astype(str).str.strip())
control_ids = set(control_df['PersonId'].dropna().astype(str).str.strip())

print("Prenatal unique ids:", len(prenatal_ids))
print("control_df unique ids:", len(control_ids))
print("overlap:", len(prenatal_ids & control_ids))

In [31]:
Prenatal['RecordedDateTime'] = pd.to_datetime(Prenatal['RecordedDateTime'], errors='coerce')
control_df['estimated_LMP'] = pd.to_datetime(control_df['estimated_LMP'], errors='coerce')
control_df['delivery_date'] = pd.to_datetime(control_df['delivery_date'], errors='coerce')

In [32]:
control_df = mark_condition_in_pregnancy(Prenatal, control_df, 'obstetric_care')
control_df.obstetric_care.value_counts()

In [33]:
control_df = mark_condition_in_pregnancy(csection, control_df, 'csection1')
control_df.csection.value_counts()

In [34]:
control_df.csection.value_counts()

In [31]:
control_df.shape, control_df.PersonId.nunique()

In [22]:
control.gestation_weight.describe()

### GWG - normal distribution plot

In [31]:
import matplotlib.pyplot as plt
# Plot histogram
c_gwg = control["gestation_weight"]
plt.hist(c_gwg, bins=40, edgecolor='black')
mean_val = c_gwg.mean()
median_val = c_gwg.median()
plt.title("Histogram of Control Group Gestation Weight Gain")
plt.xlabel("Gestation Weight Gain")
plt.ylabel("Frequency")
# Add vertical lines
plt.axvline(mean_val, linestyle='--', label=f"Mean = {mean_val:.2f}")
plt.axvline(median_val, linestyle='-', label=f"Median = {median_val:.2f}")
plt.legend()
plt.show()

In [32]:
t_gwg = test["gestation_weight"]
plt.hist(t_gwg, bins=40, edgecolor='black')
mean_val = t_gwg.mean()
median_val = t_gwg.median()
plt.title("Histogram of Medication Group Gestation Weight Gain")
plt.xlabel("Gestation Weight Gain")
plt.ylabel("Frequency")
plt.axvline(mean_val, linestyle='--', label=f"Mean = {mean_val:.2f}")
plt.axvline(median_val, linestyle='-', label=f"Median = {median_val:.2f}")
plt.legend()
plt.show()

In [33]:
test.columns

### GWG - categorize GWG - Don't Run

In [22]:
file_to_read = output_path_local + "/weights_test.csv"
weights_test = pd.read_csv(file_to_read)
weights_test.head()

In [28]:
weight_df = weights_test.merge(
    df[["PersonId", "delivery_date", "estimated_LMP"]],
    on="PersonId",
    how="inner"
)
weight_df["RecordedDateTime"] = pd.to_datetime(weight_df["RecordedDateTime"], errors="coerce")
weight_df["delivery_date"] = pd.to_datetime(weight_df["delivery_date"], errors="coerce")
weight_df["estimated_LMP"] = pd.to_datetime(weight_df["estimated_LMP"], errors="coerce")
weight_df = weight_df[
    (weight_df["RecordedDateTime"] >= weight_df["estimated_LMP"]) &
    (weight_df["RecordedDateTime"] <= weight_df["delivery_date"])
].copy()

weight_df["diff_days"] = (
    weight_df["delivery_date"] - weight_df["RecordedDateTime"]
).dt.days

weight_df = weight_df[
    (weight_df["diff_days"] >= 0) &
    (weight_df["diff_days"] <= 28)
].copy()

idx = weight_df.groupby(["PersonId", "delivery_date"])["RecordedDateTime"].idxmax()
last_weight_df = weight_df.loc[idx].copy()

In [32]:
# -----------------------------
# A. BMI category
# -----------------------------
def bmi_category(bmi):
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25.0:
        return "Normal weight"
    elif bmi < 30.0:
        return "Overweight"
    else:
        return "Obese"


# -----------------------------
# B. Calculate recommended GWG bounds
#    weeks = gestational week at last weight
# -----------------------------
def gwg_bounds(bmi, weeks):
    if pd.isna(bmi) or pd.isna(weeks):
        return (np.nan, np.nan)

    # safeguard for weights recorded before 13 weeks
    delta_weeks = max(weeks - 13, 0)

    if bmi < 18.5:  # Underweight
        low = 0.5 + 0.44 * delta_weeks
        high = 2.0 + 0.58 * delta_weeks
    elif bmi < 25.0:  # Normal weight
        low = 0.5 + 0.35 * delta_weeks
        high = 2.0 + 0.50 * delta_weeks
    elif bmi < 30.0:  # Overweight
        low = 0.5 + 0.23 * delta_weeks
        high = 2.0 + 0.33 * delta_weeks
    else:  # Obese
        low = 0.5 + 0.17 * delta_weeks
        high = 2.0 + 0.27 * delta_weeks

    return (low, high)


# -----------------------------
# C. Categorize GWG
# -----------------------------
def categorize_gwg(gwg, low, high):
    if pd.isna(gwg) or pd.isna(low) or pd.isna(high):
        return np.nan
    if gwg < low:
        return "Inadequate"
    elif gwg <= high:
        return "Adequate"
    else:
        return "Excessive"

In [45]:
# =========================================
# STEP 2. Calculate GWG and categorize it
# =========================================

# ---- keep pregnancies with prepregnancy weight and BMI ----
analysis_df = last_weight_df.merge(
    df[["PersonId", "prepreg_weight", "PrePregnancyBMI", 'user_group']], on = "PersonId", how = "left"
).copy()

# ---- calculate observed GWG (kg) ----
analysis_df["GWG"] = (
    analysis_df["last_weight_kg"] - analysis_df["prepreg_weight"]
)

# ---- BMI group ----
analysis_df["bmi_group"] = analysis_df["PrePregnancyBMI"].apply(bmi_category)

# ---- recommended GWG bounds ----
analysis_df[["gwg_low", "gwg_high"]] = analysis_df.apply(
    lambda row: pd.Series(
        gwg_bounds(
            row["PrePregnancyBMI"],
            row["gestational_week_last_weight"]
        )
    ),
    axis=1
)

# ---- categorize GWG ----
analysis_df["gwg_category"] = analysis_df.apply(
    lambda row: categorize_gwg(
        row["GWG"],
        row["gwg_low"],
        row["gwg_high"]
    ),
    axis=1
)

# ---- optional: keep plausible gestational week at last weight ----
analysis_df = analysis_df[
    (analysis_df["gestational_week_last_weight"] >= 24) &
    (analysis_df["gestational_week_last_weight"] <= 42)
].copy()

print("Final analysis_df shape:", analysis_df.shape)
print(analysis_df["GWG"].describe())
print(analysis_df["gwg_category"].value_counts(dropna=False))
print(analysis_df["gwg_category"].value_counts(normalize=True, dropna=False))

In [48]:
from tableone import TableOne

columns = ['gwg_category']
categorical = ['gwg_category']
groupby = 'user_group'

table1 = TableOne(
    analysis_df,
    columns=columns,
    categorical=categorical,
    groupby=groupby,
    pval=True 
)

table1

In [34]:
import pandas as pd
import numpy as np

# -----------------------------
# 1. Define BMI category
# -----------------------------
def bmi_category(bmi):
    if pd.isna(bmi):
        return np.nan
    if bmi < 18.5:
        return "Underweight"
    elif 18.5 <= bmi < 25.0:
        return "Normal weight"
    elif 25.0 <= bmi < 30.0:
        return "Overweight"
    else:
        return "Obese"

# -----------------------------
# 2. Calculate recommended GWG bounds
# -----------------------------
def gwg_bounds(bmi, weeks):
    """
    Returns (low_bound, high_bound) based on IOM weekly GWG recommendation.
    Formula uses:
        low = first_trimester_low + weekly_low * (weeks - 13)
        high = first_trimester_high + weekly_high * (weeks - 13)
    """
    if pd.isna(bmi) or pd.isna(weeks):
        return (np.nan, np.nan)

    # Optional safeguard: weeks earlier than 13
    # If you want to strictly apply formula as written, remove max(..., 0)
    delta_weeks = max(weeks - 13, 0)

    if bmi < 18.5:  # Underweight
        low = 0.5 + 0.44 * delta_weeks
        high = 2.0 + 0.58 * delta_weeks
    elif 18.5 <= bmi < 25.0:  # Normal weight
        low = 0.5 + 0.35 * delta_weeks
        high = 2.0 + 0.50 * delta_weeks
    elif 25.0 <= bmi < 30.0:  # Overweight
        low = 0.5 + 0.23 * delta_weeks
        high = 2.0 + 0.33 * delta_weeks
    else:  # Obese
        low = 0.5 + 0.17 * delta_weeks
        high = 2.0 + 0.27 * delta_weeks

    return (low, high)

# -----------------------------
# 3. Categorize GWG
# -----------------------------
def categorize_gwg(gwg, low, high):
    if pd.isna(gwg) or pd.isna(low) or pd.isna(high):
        return np.nan
    if gwg < low:
        return "Inadequate"
    elif low <= gwg <= high:
        return "Adequate"
    else:
        return "Excessive"

# -----------------------------
# 4. Apply to dataframe
# -----------------------------
bmi_col = "PrePregnancyBMI"
weeks_col = "gestational_week"
gwg_col = "gestation_weight"

In [36]:
test["bmi_category"] = test[bmi_col].apply(bmi_category)

bounds = test.apply(lambda row: gwg_bounds(row[bmi_col], row[weeks_col]), axis=1)
test[["gwg_low_bound", "gwg_high_bound"]] = pd.DataFrame(bounds.tolist(), index=test.index)

test["gwg_category"] = test.apply(
    lambda row: categorize_gwg(row[gwg_col], row["gwg_low_bound"], row["gwg_high_bound"]),
    axis=1
)

# Optional: make category ordered
test["gwg_category"] = pd.Categorical(
    test["gwg_category"],
    categories=["Inadequate", "Adequate", "Excessive"],
    ordered=True
)

test[[bmi_col, weeks_col, gwg_col, "bmi_category", "gwg_low_bound", "gwg_high_bound", "gwg_category"]].head()

In [37]:
test.bmi_category.value_counts(), test.gwg_category.value_counts()

In [38]:
control["bmi_category"] = control[bmi_col].apply(bmi_category)

bounds = control.apply(lambda row: gwg_bounds(row[bmi_col], row[weeks_col]), axis=1)
control[["gwg_low_bound", "gwg_high_bound"]] = pd.DataFrame(bounds.tolist(), index=control.index)

control["gwg_category"] = control.apply(
    lambda row: categorize_gwg(row[gwg_col], row["gwg_low_bound"], row["gwg_high_bound"]),
    axis=1
)

# Optional: make category ordered
control["gwg_category"] = pd.Categorical(
    control["gwg_category"],
    categories=["Inadequate", "Adequate", "Excessive"],
    ordered=True
)

control.bmi_category.value_counts(), control.gwg_category.value_counts()

### Save the update datasets - Don't Run

In [39]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/test_t3.csv"
test.to_csv(file_to_write, index = False)

file_to_write = output_path_local + "/control_t1.csv"
control.to_csv(file_to_write, index = False)

In [35]:
file_to_write = output_path_local + "/control_t1.csv"
control_df.to_csv(file_to_write, index = False)

In [42]:
med = test[test["source_type"] == "med"]
med.head()

### Comparison Test

In [44]:
cols = [
    'pregnancy_exposure_group',
    'user_group',
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm',
    'supply_days',
    "obstetric_care"
]

In [45]:
def ensure_columns(df, columns):
    for col in columns:
        if col not in df.columns:
            df[col] = np.nan
    return df

control_df = ensure_columns(control_df, cols)

### match propensity score

In [46]:
ps_covariates = [
    'age_at_delivery',
    'race_ethnicity',
    'PrePregnancyBMI',
    't2d_before_pregnancy',
    'hyper_before_pregnancy'
]

df_treated_sub = df[cols].copy()
df_control_sub = control_df[cols].copy()
df_all = pd.concat([df_treated_sub, df_control_sub], ignore_index=True)

In [47]:
df_treated_sub.shape

In [48]:
df_all.user_group.value_counts()

##### pms + table 2

##### Table 2

In [49]:
df_all['gwg_excessive_flag'] = (df_all['gwg_category'] == 'Excessive').map({
    True: 'yes',
    False: 'no'
})

In [50]:
cols = [
    'pregnancy_exposure_group',
    'user_group',
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm',
    'supply_days',
    "obstetric_care",
    'gwg_excessive_flag',
]

In [51]:
df_all.columns

In [52]:
import pandas as pd
import numpy as np
from pathlib import Path

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression
from scipy.special import logit
from tableone import TableOne


# =========================================================
# 0. User settings
# =========================================================

group_col = 'user_group'

# only for propensity score matching
ps_covariates = [
    'age_at_delivery',
    'race_ethnicity',
    'PrePregnancyBMI',
    't2d_before_pregnancy',
    'hyper_before_pregnancy'
]

# extra adjustment in outcome model
outcome_extra_covariates = [
    'prior_Csection',
    'Prior_Preterm_Birth'
]

continuous_outcomes = [
    'gestation_weight'
]

binary_outcomes = [
    'gwg_excessive_flag',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

all_outcomes = continuous_outcomes + binary_outcomes

pairwise_comparisons = [
    ('continued_user', 'former_user', 'continued_vs_former'),
    ('continued_user', 'non_user', 'continued_vs_non'),
    ('former_user', 'non_user', 'former_vs_non')
]

# IMPORTANT: keep extra columns for Table 1 in matched output
extra_keep_cols = [
    'pregnancy_exposure_group',
    'preTreatmentBMI',
    'weight_loss',
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    'supply_days',
    'obstetric_care',
    'zcode_count',
    'gestational_week',
    'gwg_category'
]

# output path
output_dir = Path(output_path_local) / "revise_results"
output_dir.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. Helpers
# =========================================================

def clean_binary_series(s):
    if s.dtype == bool:
        return s.astype(int)

    mapping = {
        'yes': 1, 'no': 0,
        'y': 1, 'n': 0,
        'true': 1, 'false': 0,
        't': 1, 'f': 0,
        'case': 1, 'control': 0,
        'positive': 1, 'negative': 0
    }

    if s.dtype == object or str(s.dtype).startswith("string"):
        s2 = s.astype(str).str.strip().str.lower()
        s2 = s2.replace(mapping)
        s2 = pd.to_numeric(s2, errors='coerce')
        return s2

    return pd.to_numeric(s, errors='coerce')


def safe_numeric(s):
    return pd.to_numeric(s, errors='coerce')


def ensure_columns(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


# =========================================================
# 2. Prepare pairwise data for PS matching
# =========================================================

def prepare_pairwise_data(df_all, group_col, group1, group0, covariates,
                          outcome_extra_covariates, outcomes, extra_keep_cols=None):
    df_sub = df_all[df_all[group_col].isin([group1, group0])].copy()

    if df_sub.empty:
        return pd.DataFrame()

    df_sub['treat'] = (df_sub[group_col] == group1).astype(int)

    id_fields = ['PersonId'] if 'PersonId' in df_sub.columns else []

    needed = list(dict.fromkeys(
        id_fields +
        [group_col, 'treat'] +
        covariates +
        outcome_extra_covariates +
        outcomes +
        ['Income', 'parity', 'depression'] +
        (extra_keep_cols if extra_keep_cols is not None else [])
    ))

    df_sub = ensure_columns(df_sub, needed)
    df_sub = df_sub[needed].copy()

    df_sub = df_sub.reset_index(drop=False).rename(columns={'index': 'orig_index'})
    return df_sub


def build_ps_design_matrix(df_sub, covariates):
    X = df_sub[covariates].copy()

    if 'race_ethnicity' in X.columns:
        X['race_ethnicity'] = X['race_ethnicity'].fillna('Unknown').astype(str)

    for c in X.columns:
        if c == 'race_ethnicity':
            continue
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)
        else:
            X[c] = pd.to_numeric(X[c], errors='coerce')

    numeric_cols = [c for c in X.columns if c != 'race_ethnicity']
    for c in numeric_cols:
        if X[c].isna().all():
            X[c] = 0
        else:
            X[c] = X[c].fillna(X[c].median())

    if 'race_ethnicity' in X.columns:
        X = pd.get_dummies(X, columns=['race_ethnicity'], drop_first=True)

    for c in X.columns:
        if X[c].dtype == bool:
            X[c] = X[c].astype(int)

    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors='coerce')

    for c in X.columns:
        if X[c].isna().all():
            X[c] = 0
        else:
            X[c] = X[c].fillna(X[c].median())

    return X


def run_ps(df_sub, covariates):
    if df_sub.empty:
        raise ValueError("run_ps received an empty dataframe.")

    X = build_ps_design_matrix(df_sub, covariates)
    y = df_sub['treat'].copy()

    valid_mask = np.isfinite(X).all(axis=1)
    X = X.loc[valid_mask].copy()
    y = y.loc[valid_mask].copy()
    df_sub = df_sub.loc[valid_mask].copy()

    if X.empty:
        raise ValueError("No rows remain after building the PS design matrix.")

    if y.nunique() < 2:
        raise ValueError("Treatment indicator has fewer than 2 classes after filtering.")

    model = LogisticRegression(max_iter=2000)
    model.fit(X, y)

    df_sub = df_sub.copy()
    df_sub['ps'] = model.predict_proba(X)[:, 1]
    return df_sub, X.columns.tolist(), model


def compute_caliper_from_logit_ps(df_sub, multiplier=0.2):
    ps = df_sub['ps'].clip(1e-6, 1 - 1e-6)
    return multiplier * np.std(logit(ps))


def match_ps_without_replacement(df_sub, caliper=None, caliper_type='logit_ps'):
    if df_sub.empty:
        return pd.DataFrame()

    treated = df_sub[df_sub['treat'] == 1].copy().sort_values('ps')
    control = df_sub[df_sub['treat'] == 0].copy().sort_values('ps')

    if treated.empty or control.empty:
        return pd.DataFrame()

    if caliper_type == 'logit_ps':
        treated['match_score'] = logit(treated['ps'].clip(1e-6, 1 - 1e-6))
        control['match_score'] = logit(control['ps'].clip(1e-6, 1 - 1e-6))
    else:
        treated['match_score'] = treated['ps']
        control['match_score'] = control['ps']

    control_available = control.copy()
    matched_rows = []
    pair_id = 0

    for _, t_row in treated.iterrows():
        if control_available.empty:
            break

        control_available = control_available.copy()
        control_available['score_diff'] = (
            control_available['match_score'] - t_row['match_score']
        ).abs()

        best_idx = control_available['score_diff'].idxmin()
        best_diff = control_available.loc[best_idx, 'score_diff']

        if (caliper is None) or (best_diff <= caliper):
            t_out = t_row.copy()
            c_out = control_available.loc[best_idx].copy()

            t_out['pair_id'] = pair_id
            c_out['pair_id'] = pair_id

            matched_rows.append(t_out)
            matched_rows.append(c_out)

            control_available = control_available.drop(index=best_idx)
            pair_id += 1

    matched_df = pd.DataFrame(matched_rows).reset_index(drop=True)

    if 'score_diff' in matched_df.columns:
        matched_df = matched_df.drop(columns=['score_diff'])

    return matched_df


# =========================================================
# 3. Balance check
# =========================================================

def smd_continuous(x_t, x_c):
    x_t = pd.to_numeric(pd.Series(x_t), errors='coerce').dropna()
    x_c = pd.to_numeric(pd.Series(x_c), errors='coerce').dropna()

    if len(x_t) == 0 or len(x_c) == 0:
        return np.nan

    var_t = np.var(x_t, ddof=1) if len(x_t) > 1 else 0
    var_c = np.var(x_c, ddof=1) if len(x_c) > 1 else 0
    pooled_sd = np.sqrt((var_t + var_c) / 2)

    if pooled_sd == 0 or np.isnan(pooled_sd):
        return 0.0
    return abs(np.mean(x_t) - np.mean(x_c)) / pooled_sd


def smd_binary(x_t, x_c):
    x_t = pd.to_numeric(pd.Series(x_t), errors='coerce').dropna()
    x_c = pd.to_numeric(pd.Series(x_c), errors='coerce').dropna()

    if len(x_t) == 0 or len(x_c) == 0:
        return np.nan

    p1 = np.mean(x_t)
    p0 = np.mean(x_c)
    denom = np.sqrt((p1 * (1 - p1) + p0 * (1 - p0)) / 2)

    if denom == 0 or np.isnan(denom):
        return 0.0
    return abs(p1 - p0) / denom


def get_balance_table(matched_df, covariates):
    if matched_df.empty:
        return pd.DataFrame(columns=['variable', 'SMD'])

    X = build_ps_design_matrix(matched_df, covariates)
    balance_df = pd.concat(
        [matched_df[['treat']].reset_index(drop=True), X.reset_index(drop=True)],
        axis=1
    )

    treated = balance_df[balance_df['treat'] == 1]
    control = balance_df[balance_df['treat'] == 0]

    rows = []
    for col in X.columns:
        vals = balance_df[col].dropna().unique()
        if len(vals) == 0:
            smd_val = np.nan
        elif set(vals).issubset({0, 1}):
            smd_val = smd_binary(treated[col], control[col])
        else:
            smd_val = smd_continuous(treated[col], control[col])

        rows.append({'variable': col, 'SMD': smd_val})

    return pd.DataFrame(rows).sort_values('SMD', ascending=False).reset_index(drop=True)


# =========================================================
# 4. Recode matched data for outcome models
# =========================================================

def prepare_matched_for_models(matched_df, treat_label, ref_label):
    if matched_df.empty:
        return matched_df.copy()

    df = matched_df.copy()

    df['group'] = np.where(df['treat'] == 1, treat_label, ref_label)

    if 'race_ethnicity' in df.columns:
        df['race_ethnicity'] = df['race_ethnicity'].fillna("Unknown")
        df['race_ethnicity'] = pd.Categorical(
            df['race_ethnicity'],
            categories=[
                "Non-Hispanic White",
                "Non-Hispanic Black",
                "Hispanic",
                "Other",
                "Unknown"
            ],
            ordered=True
        )

    if 'Income' in df.columns:
        df['Income'] = df['Income'].fillna("Unknown")
        df['Income'] = pd.Categorical(
            df['Income'],
            categories=["≤50000", "50001-80000", ">80000", "Unknown"],
            ordered=True
        )

    if 'parity' in df.columns:
        df['parity'] = df['parity'].fillna("Unknown")
        df['parity'] = pd.Categorical(
            df['parity'],
            categories=["Primiparous", "Multiparous", "Unknown"],
            ordered=True
        )

    df['group'] = pd.Categorical(
        df['group'],
        categories=[ref_label, treat_label],
        ordered=True
    )

    for col in binary_outcomes:
        if col in df.columns:
            df[col] = clean_binary_series(df[col])

    numeric_cols = [
        'age_at_delivery',
        'PrePregnancyBMI',
        't2d_before_pregnancy',
        'hyper_before_pregnancy',
        'depression',
        'prior_Csection',
        'Prior_Preterm_Birth'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = safe_numeric(df[col])

    return df


# =========================================================
# 5. Outcome model helpers
# =========================================================

def get_covariates_for_outcome(outcome, df_subset):
    covs = [
        "C(group)",
        "age_at_delivery",
        "C(race_ethnicity)",
        "C(Income)",
        "PrePregnancyBMI",
        "C(parity)",
        "t2d_before_pregnancy",
        "hyper_before_pregnancy",
        "depression",
        "prior_Csection",
        "Prior_Preterm_Birth"
    ]

    if outcome == "gest_diabetes_no_prior_t2d":
        covs = [c for c in covs if "t2d_before_pregnancy" not in c]

    if outcome in ["gest_hyper_no_prior_hyper", "preg_related_htn"]:
        covs = [c for c in covs if "hyper_before_pregnancy" not in c]

    valid_covariates = []
    for cov in covs:
        colname = cov.split("(")[-1].split(")")[0] if "C(" in cov else cov
        if colname in df_subset.columns and df_subset[colname].nunique(dropna=True) > 1:
            valid_covariates.append(cov)

    return valid_covariates


def extract_model_results(model, outcome_name, comparison_name, model_type="logit", fit_engine=None):
    conf_int = model.conf_int()
    summary_df = pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err": model.bse.values,
        "z_or_t": model.tvalues if hasattr(model, "tvalues") else model.params / model.bse,
        "p_value": model.pvalues.values,
        "ci_lower": conf_int[0].values,
        "ci_upper": conf_int[1].values
    })

    if model_type == "logit":
        summary_df["odds_ratio"] = np.exp(summary_df["coef"])
        summary_df["or_ci_lower"] = np.exp(summary_df["ci_lower"])
        summary_df["or_ci_upper"] = np.exp(summary_df["ci_upper"])
        summary_df["formatted"] = summary_df.apply(
            lambda row: f"OR={row['odds_ratio']:.2f}, ({row['or_ci_lower']:.2f}, {row['or_ci_upper']:.2f}), p={row['p_value']:.3f}",
            axis=1
        )
    else:
        summary_df["formatted"] = summary_df.apply(
            lambda row: f"β={row['coef']:.2f}, ({row['ci_lower']:.2f}, {row['ci_upper']:.2f}), p={row['p_value']:.3f}",
            axis=1
        )

    summary_df["outcome"] = outcome_name
    summary_df["comparison"] = comparison_name
    summary_df["model_type"] = model_type
    summary_df["fit_engine"] = fit_engine

    return summary_df


def run_models(df, outcomes, comparison_name, model_type="logit", output_csv=None, status_csv=None):
    all_results = []
    model_status_rows = []

    if df.empty:
        status_df = pd.DataFrame([{
            "comparison": comparison_name,
            "outcome": None,
            "model_type": model_type,
            "status": "skipped_empty_input_df",
            "n": 0,
            "n_event": np.nan,
            "message": "Input dataframe is empty"
        }])

        if status_csv is not None:
            status_df.to_csv(status_csv, index=False)

        final_df = pd.DataFrame()
        if output_csv is not None:
            final_df.to_csv(output_csv, index=False)

        return final_df, status_df

    for outcome in outcomes:
        if outcome not in df.columns:
            model_status_rows.append({
                "comparison": comparison_name,
                "outcome": outcome,
                "model_type": model_type,
                "status": "skipped_missing_column",
                "n": np.nan,
                "n_event": np.nan,
                "message": "Outcome column not found"
            })
            continue

        df_subset = df.copy()
        covariates = get_covariates_for_outcome(outcome, df_subset)

        cols_needed = [outcome] + [
            c.split("(")[-1].split(")")[0] if "C(" in c else c
            for c in covariates
        ]
        cols_needed = [c for c in cols_needed if c in df_subset.columns]

        df_subset = df_subset.dropna(subset=cols_needed).copy()

        if outcome in binary_outcomes:
            df_subset[outcome] = clean_binary_series(df_subset[outcome])

        if df_subset.empty:
            model_status_rows.append({
                "comparison": comparison_name,
                "outcome": outcome,
                "model_type": model_type,
                "status": "skipped_empty_after_dropna",
                "n": 0,
                "n_event": np.nan,
                "message": "No rows left after dropna"
            })
            continue

        nunique = df_subset[outcome].nunique(dropna=True)
        n_total = len(df_subset)
        n_event = df_subset[outcome].sum() if outcome in binary_outcomes else np.nan

        if nunique < 2:
            model_status_rows.append({
                "comparison": comparison_name,
                "outcome": outcome,
                "model_type": model_type,
                "status": "skipped_no_variation",
                "n": n_total,
                "n_event": n_event,
                "message": "Outcome has <2 unique values"
            })
            continue

        formula = f"{outcome} ~ " + (" + ".join(covariates) if len(covariates) > 0 else "1")

        try:
            if model_type == "logit":
                try:
                    model = smf.logit(formula=formula, data=df_subset).fit(disp=False)
                    fit_engine = "logit"
                except Exception:
                    model = smf.glm(
                        formula=formula,
                        data=df_subset,
                        family=sm.families.Binomial()
                    ).fit()
                    fit_engine = "glm_binomial"

            elif model_type == "ols":
                model = smf.ols(formula=formula, data=df_subset).fit()
                fit_engine = "ols"
            else:
                raise ValueError("Unsupported model_type")

            result_df = extract_model_results(
                model=model,
                outcome_name=outcome,
                comparison_name=comparison_name,
                model_type=model_type,
                fit_engine=fit_engine
            )
            all_results.append(result_df)

            model_status_rows.append({
                "comparison": comparison_name,
                "outcome": outcome,
                "model_type": model_type,
                "status": "fitted",
                "n": n_total,
                "n_event": n_event,
                "message": fit_engine
            })

        except Exception as e:
            model_status_rows.append({
                "comparison": comparison_name,
                "outcome": outcome,
                "model_type": model_type,
                "status": "error",
                "n": n_total,
                "n_event": n_event,
                "message": str(e)
            })

    final_df = pd.concat(all_results, ignore_index=True) if len(all_results) > 0 else pd.DataFrame()
    status_df = pd.DataFrame(model_status_rows)

    if output_csv is not None:
        final_df.to_csv(output_csv, index=False)

    if status_csv is not None:
        status_df.to_csv(status_csv, index=False)

    return final_df, status_df


# =========================================================
# 6. Run full pipeline for one pair
# =========================================================

def empty_result_package(label, group1, group0, message):
    return {
        "df_ps": pd.DataFrame(),
        "matched_df": pd.DataFrame(),
        "matched_model_df": pd.DataFrame(),
        "balance_table": pd.DataFrame(),
        "logit_results": pd.DataFrame(),
        "ols_results": pd.DataFrame(),
        "logit_status": pd.DataFrame(),
        "ols_status": pd.DataFrame(),
        "summary": {
            "comparison": label,
            "group1": group1,
            "group0": group0,
            "n_before_treated": 0,
            "n_before_control": 0,
            "n_after_treated": 0,
            "n_after_control": 0,
            "n_pairs": 0,
            "caliper": np.nan,
            "max_smd": np.nan,
            "message": message
        }
    }


def run_pair_analysis(df_all, group1, group0, label):
    outcomes = continuous_outcomes + binary_outcomes

    df_sub = prepare_pairwise_data(
        df_all=df_all,
        group_col=group_col,
        group1=group1,
        group0=group0,
        covariates=ps_covariates,
        outcome_extra_covariates=outcome_extra_covariates,
        outcomes=outcomes,
        extra_keep_cols=extra_keep_cols
    )

    if df_sub.empty:
        return empty_result_package(label, group1, group0, "No rows after pairwise subsetting")

    try:
        df_ps, ps_features, ps_model = run_ps(df_sub, ps_covariates)
    except Exception as e:
        return empty_result_package(label, group1, group0, f"PS model failed: {e}")

    caliper = compute_caliper_from_logit_ps(df_ps, multiplier=0.2)

    matched_df = match_ps_without_replacement(
        df_ps,
        caliper=caliper,
        caliper_type='logit_ps'
    )

    if matched_df.empty:
        return empty_result_package(label, group1, group0, "No matched pairs found")

    balance_tbl = get_balance_table(matched_df, ps_covariates)

    matched_model_df = prepare_matched_for_models(
        matched_df=matched_df,
        treat_label=group1,
        ref_label=group0
    )

    logit_csv = output_dir / f"{label}_logit_results.csv"
    ols_csv = output_dir / f"{label}_ols_results.csv"
    balance_csv = output_dir / f"{label}_balance.csv"
    matched_csv = output_dir / f"{label}_matched.csv"
    logit_status_csv = output_dir / f"{label}_logit_model_status.csv"
    ols_status_csv = output_dir / f"{label}_ols_model_status.csv"

    logit_results, logit_status = run_models(
        matched_model_df,
        binary_outcomes,
        comparison_name=label,
        model_type="logit",
        output_csv=logit_csv,
        status_csv=logit_status_csv
    )

    ols_results, ols_status = run_models(
        matched_model_df,
        continuous_outcomes,
        comparison_name=label,
        model_type="ols",
        output_csv=ols_csv,
        status_csv=ols_status_csv
    )

    balance_tbl.to_csv(balance_csv, index=False)
    matched_model_df.to_csv(matched_csv, index=False)

    summary = {
        "comparison": label,
        "group1": group1,
        "group0": group0,
        "n_before_treated": int((df_ps["treat"] == 1).sum()),
        "n_before_control": int((df_ps["treat"] == 0).sum()),
        "n_after_treated": int((matched_df["treat"] == 1).sum()),
        "n_after_control": int((matched_df["treat"] == 0).sum()),
        "n_pairs": int(matched_df["pair_id"].nunique()) if "pair_id" in matched_df.columns else 0,
        "caliper": caliper,
        "max_smd": balance_tbl["SMD"].max() if len(balance_tbl) > 0 else np.nan
    }

    return {
        "df_ps": df_ps,
        "matched_df": matched_df,
        "matched_model_df": matched_model_df,
        "balance_table": balance_tbl,
        "logit_results": logit_results,
        "ols_results": ols_results,
        "logit_status": logit_status,
        "ols_status": ols_status,
        "summary": summary
    }


# =========================================================
# 7. Run all pairwise comparisons
# =========================================================

print("Checking whether extra_keep_cols exist in df_all...")
missing_from_df_all = [c for c in extra_keep_cols if c not in df_all.columns]
if missing_from_df_all:
    print("These columns are not in df_all and will be all-NaN in matched output:")
    print(missing_from_df_all)
else:
    print("All extra_keep_cols are present in df_all.")

analysis_results = {}

for group1, group0, label in pairwise_comparisons:
    print(f"\nRunning: {label}")
    analysis_results[label] = run_pair_analysis(df_all, group1, group0, label)

summary_df = pd.DataFrame([v["summary"] for v in analysis_results.values()])
summary_df.to_csv(output_dir / "pairwise_summary.csv", index=False)

print("\n=== Pairwise summary ===")
print(summary_df)


# =========================================================
# 8. Extract group effect only
# =========================================================

def extract_group_effect_only(results_df):
    if results_df.empty:
        return results_df

    out = results_df[
        results_df["variable"].str.contains(r"C\(group\)", regex=True, na=False)
    ].copy()

    keep_cols = [
        "comparison", "outcome", "variable", "formatted",
        "model_type", "fit_engine", "p_value"
    ]
    keep_cols = [c for c in keep_cols if c in out.columns]
    return out[keep_cols]


group_effect_tables = []

for label, res in analysis_results.items():
    if not res["logit_results"].empty:
        group_effect_tables.append(extract_group_effect_only(res["logit_results"]))
    if not res["ols_results"].empty:
        group_effect_tables.append(extract_group_effect_only(res["ols_results"]))

group_effect_table = (
    pd.concat(group_effect_tables, ignore_index=True)
    if len(group_effect_tables) > 0 else pd.DataFrame()
)
group_effect_table.to_csv(output_dir / "group_effect_only_summary.csv", index=False)


# =========================================================
# 9. Combine model status tables
# =========================================================

all_status = []
for label, res in analysis_results.items():
    if "logit_status" in res and not res["logit_status"].empty:
        all_status.append(res["logit_status"])
    if "ols_status" in res and not res["ols_status"].empty:
        all_status.append(res["ols_status"])

model_status_df = pd.concat(all_status, ignore_index=True) if len(all_status) > 0 else pd.DataFrame()
model_status_df.to_csv(output_dir / "all_model_status_summary.csv", index=False)

print("\n=== Matching pipeline done ===")
print(f"Results saved to: {output_dir}")


# =========================================================
# 10. Build Table 1 from regenerated matched files
# =========================================================

files = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non": "continued_vs_non_matched.csv",
    "former_vs_non": "former_vs_non_matched.csv"
}

table1_cols = [
    'pregnancy_exposure_group',
    'user_group',
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'supply_days',
    'obstetric_care',
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

categorical = [
    'pregnancy_exposure_group',
    'user_group',
    'race_ethnicity',
    'Income',
    'parity',
    'med_indict',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

nonnormal = [
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    'supply_days',
    'gestational_week',
    'weight_loss'
]

for label, filename in files.items():
    print(f"\n===== {label} =====")

    df = pd.read_csv(output_dir / filename)
    print("Columns in file:")
    print(df.columns.tolist())

    if 'user_group' in df.columns:
        df['user_group'] = df['user_group'].astype(str)

    available_cols = [c for c in table1_cols if c in df.columns]
    available_categorical = [c for c in categorical if c in df.columns]
    available_nonnormal = [c for c in nonnormal if c in df.columns]

    missing_cols = [c for c in table1_cols if c not in df.columns]
    if missing_cols:
        print("Missing columns skipped:")
        print(missing_cols)

    for col in available_nonnormal:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if 'user_group' not in df.columns:
        print(f"Skipped {label}: user_group not found.")
        continue

    table1 = TableOne(
        data=df,
        columns=available_cols,
        categorical=available_categorical,
        groupby='user_group',
        nonnormal=available_nonnormal,
        pval=True
    )

    table1.to_csv(output_dir / f"{label}_table1_tableone.csv")
    print(table1)

print("\n=== TableOne done ===")

##### Unadjust model

In [34]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

# ==============================
# 1. 路径
# ==============================

data_dir = Path(output_path_local) / "revise_results"

files = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non": "continued_vs_non_matched.csv",
    "former_vs_non": "former_vs_non_matched.csv"
}

# 为每个 comparison 明确指定 reference group
reference_map = {
    "continued_vs_former": "former_user",
    "continued_vs_non": "non_user",
    "former_vs_non": "non_user"
}

# 可选：也把 treat group 明确写出来，方便控制 category 顺序
treat_map = {
    "continued_vs_former": "continued_user",
    "continued_vs_non": "continued_user",
    "former_vs_non": "former_user"
}

# ==============================
# 2. outcome 定义
# ==============================

continuous_outcomes = [
    'gestation_weight'
]

binary_outcomes = [
    'gwg_excessive_flag',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

# 这里是你想额外加入 comparison model 的 covariates
extra_covariates = [
    'prior_Csection',
    'Prior_Preterm_Birth'
]

# ==============================
# 3. 辅助函数
# ==============================

def ensure_columns(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df


def extract_adjusted_results(model, outcome_name, comparison_name, model_type):
    result_df = pd.DataFrame({
        "variable": model.params.index,
        "coef": model.params.values,
        "p_value": model.pvalues.values,
        "ci_lower": model.conf_int()[0].values,
        "ci_upper": model.conf_int()[1].values
    })

    if model_type == "logit":
        result_df["odds_ratio"] = np.exp(result_df["coef"])
        result_df["or_ci_lower"] = np.exp(result_df["ci_lower"])
        result_df["or_ci_upper"] = np.exp(result_df["ci_upper"])
        result_df["formatted"] = result_df.apply(
            lambda r: f"OR={r['odds_ratio']:.2f}, ({r['or_ci_lower']:.2f}, {r['or_ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1
        )
    else:
        result_df["formatted"] = result_df.apply(
            lambda r: f"β={r['coef']:.2f}, ({r['ci_lower']:.2f}, {r['ci_upper']:.2f}), p={r['p_value']:.3f}",
            axis=1
        )

    result_df["outcome"] = outcome_name
    result_df["comparison"] = comparison_name
    result_df["model_type"] = f"{model_type}_adjusted"

    return result_df


def get_covariates_for_outcome(outcome, df):
    """
    当前版本：所有 outcome 都默认加入
    prior_Csection 和 Prior_Preterm_Birth
    如果以后你想按 outcome 排除某个变量，可以在这里改。
    """
    covs = [
        'prior_Csection',
        'Prior_Preterm_Birth'
    ]

    valid_covs = []
    for c in covs:
        if c in df.columns and df[c].nunique(dropna=True) > 1:
            valid_covs.append(c)

    return valid_covs


# ==============================
# 4. 核心函数
# ==============================

def run_adjusted_on_df(df, label, ref_group, treat_group=None):
    all_results = []

    if 'group' not in df.columns:
        raise ValueError(f"[{label}] missing 'group' column")

    df = df.copy()

    # 确保 covariate 列存在
    df = ensure_columns(df, extra_covariates)

    # 去掉首尾空格，避免 group 名称不匹配
    df['group'] = df['group'].astype(str).str.strip()

    # covariates 转成 numeric
    for c in extra_covariates:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # 固定 group 顺序
    if treat_group is not None:
        df['group'] = pd.Categorical(
            df['group'],
            categories=[ref_group, treat_group],
            ordered=True
        )
    else:
        df['group'] = df['group'].astype('category')

    # 显式指定 reference
    group_term = f"C(group, Treatment(reference='{ref_group}'))"

    # -------- binary outcomes --------
    for outcome in binary_outcomes:
        if outcome not in df.columns:
            print(f"[{label}] Skipped {outcome} (not found)")
            continue

        covariates = get_covariates_for_outcome(outcome, df)

        cols_needed = [outcome, 'group'] + covariates
        df_sub = df[cols_needed].copy()

        df_sub[outcome] = pd.to_numeric(df_sub[outcome], errors='coerce')
        df_sub = df_sub.dropna()

        if df_sub.empty or df_sub[outcome].nunique() < 2:
            print(f"[{label}] Skipped {outcome} (no variation or empty after dropna)")
            continue

        print(f"\n[{label}] {outcome} distribution:")
        print(pd.crosstab(df_sub['group'], df_sub[outcome], dropna=False))
        print(f"[{label}] reference group = {ref_group}")
        print(f"[{label}] covariates = {covariates}")

        try:
            rhs_terms = [group_term] + covariates
            formula = f"{outcome} ~ " + " + ".join(rhs_terms)

            model = smf.logit(formula, data=df_sub).fit(disp=False)
            res = extract_adjusted_results(model, outcome, label, "logit")
            all_results.append(res)

        except Exception as e:
            print(f"[{label}] Error in {outcome}: {e}")

        # 可选 fallback
        # try:
        #     rhs_terms = [group_term] + covariates
        #     formula = f"{outcome} ~ " + " + ".join(rhs_terms)
        #     model = smf.glm(formula=formula, data=df_sub, family=sm.families.Binomial()).fit()
        #     res = extract_adjusted_results(model, outcome, label, "logit")
        #     all_results.append(res)
        # except Exception as e:
        #     print(f"[{label}] GLM fallback error in {outcome}: {e}")

    # -------- continuous outcomes --------
    for outcome in continuous_outcomes:
        if outcome not in df.columns:
            print(f"[{label}] Skipped {outcome} (not found)")
            continue

        covariates = get_covariates_for_outcome(outcome, df)

        cols_needed = [outcome, 'group'] + covariates
        df_sub = df[cols_needed].copy()

        df_sub[outcome] = pd.to_numeric(df_sub[outcome], errors='coerce')
        df_sub = df_sub.dropna()

        if df_sub.empty or df_sub[outcome].nunique() < 2:
            print(f"[{label}] Skipped {outcome} (no variation or empty after dropna)")
            continue

        print(f"\n[{label}] {outcome} summary:")
        print(df_sub.groupby('group', observed=False)[outcome].describe())
        print(f"[{label}] reference group = {ref_group}")
        print(f"[{label}] covariates = {covariates}")

        try:
            rhs_terms = [group_term] + covariates
            formula = f"{outcome} ~ " + " + ".join(rhs_terms)

            model = smf.ols(formula, data=df_sub).fit()
            res = extract_adjusted_results(model, outcome, label, "ols")
            all_results.append(res)
        except Exception as e:
            print(f"[{label}] Error in {outcome}: {e}")

    if len(all_results) == 0:
        return pd.DataFrame()

    return pd.concat(all_results, ignore_index=True)


# ==============================
# 5. 跑三个数据集
# ==============================

all_outputs = []

for label, filename in files.items():
    file_path = data_dir / filename

    print(f"\n========== Running {label} ==========")

    df = pd.read_csv(file_path)

    ref_group = reference_map[label]
    treat_group = treat_map.get(label)

    res = run_adjusted_on_df(
        df=df,
        label=label,
        ref_group=ref_group,
        treat_group=treat_group
    )

    if not res.empty:
        all_outputs.append(res)

# ==============================
# 6. 汇总 group effect
# ==============================

if len(all_outputs) == 0:
    final_df = pd.DataFrame()
    group_effect_only = pd.DataFrame()
else:
    final_df = pd.concat(all_outputs, ignore_index=True)

    # group effect 变量名会类似：
    # C(group, Treatment(reference='non_user'))[T.former_user]
    group_effect_only = final_df[
        final_df["variable"].str.contains("group", case=False, na=False)
    ].copy()

print("\n===== FINAL ADJUSTED GROUP EFFECT =====")
if not group_effect_only.empty:
    print(group_effect_only[[
        "comparison", "outcome", "variable", "formatted", "model_type", "p_value"
    ]])
else:
    print("No group effect rows found.")

# 可选保存
# final_df.to_csv(data_dir / "adjusted_full_results_all.csv", index=False)
group_effect_only.to_csv(data_dir / "adjusted_summary_all.csv", index=False)

In [29]:
group_effect_only.to_csv(data_dir / "unadjusted_summary_all.csv", index=False)

### Table 1

#### matched

In [7]:
from tableone import TableOne
import pandas as pd
from pathlib import Path

data_dir = Path(output_path_local) / "revise_results"

files = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non": "continued_vs_non_matched.csv",
    "former_vs_non": "former_vs_non_matched.csv"
}

table1_cols = [
    'pregnancy_exposure_group',
    'user_group',
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'supply_days',
    'obstetric_care',
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

categorical = [
    'pregnancy_exposure_group',
    'user_group',
    'race_ethnicity',
    'Income',
    'parity',
    'med_indict',
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

nonnormal = [
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    'supply_days',
    'gestational_week',
    'weight_loss'
]

for label, filename in files.items():
    print(f"\n===== {label} =====")

    df = pd.read_csv(data_dir / filename)
    print("Columns in file:")
    print(df.columns.tolist())

    df['user_group'] = df['user_group'].astype(str)

    available_cols = [c for c in table1_cols if c in df.columns]
    available_categorical = [c for c in categorical if c in df.columns]
    available_nonnormal = [c for c in nonnormal if c in df.columns]

    missing_cols = [c for c in table1_cols if c not in df.columns]
    if missing_cols:
        print("Missing columns skipped:")
        print(missing_cols)

    for col in available_nonnormal:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    table1 = TableOne(
        data=df,
        columns=available_cols,
        categorical=available_categorical,
        groupby='user_group',
        nonnormal=available_nonnormal,
        pval=True
    )

    table1.to_csv(data_dir / f"{label}_table1_tableone.csv")
    print(table1)

#### preg expo group

In [36]:
from tableone import TableOne
import pandas as pd
from pathlib import Path

table1_cols = [
    # groups
    'pregnancy_exposure_group',
    'user_group',

    # demographics
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',

    # BMI / weight
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',

    # medication
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    "obstetric_care",

    # outcomes
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm',
    "new_refill"
]

categorical = [
    # groups
    'pregnancy_exposure_group',
    'user_group',

    # demographics
    'race_ethnicity',
    'Income',
    'parity',

    # medication
    'med_indict',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    "obstetric_care",

    # outcomes
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm',
    "new_refill"
]

# 这两个变量按 median [Q1, Q3] / IQR 展示
nonnormal = [
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    "supply_days",
    "gestational_week",
    'weight_loss'
]

df_preg = df_preg.copy()

# 分组变量转成 string
df_preg['pregnancy_exposure_group'] = df_preg['pregnancy_exposure_group'].astype(str)

# nonnormal 变量转成 numeric
for col in nonnormal:
    if col in df_preg.columns:
        df_preg[col] = pd.to_numeric(df_preg[col], errors='coerce')

table1_preg = TableOne(
    data=df_preg,
    columns=table1_cols,
    categorical=categorical,
    groupby='pregnancy_exposure_group',
    nonnormal=nonnormal,
    pval=True
)

table1_preg.to_csv(data_dir / "pregnancy_exposure_group_table1_tableone.csv")

table1_preg

#### umatched table1 all group

In [37]:
from tableone import TableOne
import pandas as pd
from pathlib import Path
data_dir = Path(output_path_local) / "revise_results"

table1_cols = [
    # groups
    'user_group',

    # demographics
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',

    # BMI / weight
    'preTreatmentBMI',
    'PrePregnancyBMI',
    'weight_loss',

    # medication
    'med_indict',
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',

    # outcomes
    'zcode_count',
    'gestational_week',
    'gestation_weight',
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

categorical = [
    # groups
    'user_group',

    # demographics
    'race_ethnicity',
    'Income',
    'parity',

    # medication
    'med_indict',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',

    # outcomes
    'gwg_category',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

# 这两个变量按 median [Q1, Q3] / IQR 展示
nonnormal = [
    'AccumulatedPersistenceBeforePregnancy',
    'TotalExposureDaysInPregnancy',
    "gestational_week",
    'weight_loss'
]


# 分组变量转成 string
df_all['user_group'] = df_all['user_group'].astype(str)

# nonnormal 变量转成 numeric
for col in nonnormal:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

table1_unmatch = TableOne(
    data=df_all,
    columns=table1_cols,
    categorical=categorical,
    groupby='user_group',
    nonnormal=nonnormal,
    pval=True
)

table1_unmatch.to_csv(data_dir / "unmatched_table1_tableone.csv")

print(table1_unmatch)

#### umatched - cont. vs former; cont. vs non

In [38]:
df_cf = df_all[df_all['user_group'].isin(['continued_user', 'former_user'])].copy()

table1_cf = TableOne(
    data=df_cf,
    columns=table1_cols,
    categorical=categorical,
    groupby='user_group',
    nonnormal=nonnormal,
    pval=True
)

table1_cf.to_csv(data_dir / "um_table1_continued_vs_former.csv")
table1_cf

In [39]:
df_cn = df_all[df_all['user_group'].isin(['continued_user', 'non_user'])].copy()

table1_cn = TableOne(
    data=df_cn,
    columns=table1_cols,
    categorical=categorical,
    groupby='user_group',
    nonnormal=nonnormal,
    pval=True
)

table1_cn.to_csv(data_dir / "table1_continued_vs_non.csv")
table1_cn

### compare with those without weight

In [40]:
output_path_local = study.get_output_path(fs = True)
file_to_write = output_path_local + "/full_control_df.csv"
full_control_df = pd.read_csv(file_to_write)
df_control_sub["weights"] = 'With_Weight'
full_control_df["weights"] = 'Without_Weight'

file_to_write = output_path_local + "/full_med_wo_weight.csv"
full_med_df = pd.read_csv(file_to_write)
df_treated_sub["weights"] = 'With_Weight'
full_med_df["weights"] = 'Without_Weight'

In [41]:
full_med_df.obstetric_care.value_counts()

In [42]:
full_control_df.Obstetriccare.value_counts()

In [43]:
full_med_df['preg_related_htn'] = (
    (full_med_df['gest_hyper_no_prior_hyper'] == True) |
    (full_med_df['preeclampsia_no_prior_hyper'] == True)
)

full_control_df['preg_related_htn'] = (
    (full_control_df['gest_hyper_no_prior_hyper'] == True) |
    (full_control_df['preeclampsia_no_prior_hyper'] == True)
)

In [44]:
file_to_write = output_path_local + "/contrl_zcodecount.csv"
contrl_zcodecount = pd.read_csv(file_to_write)
full_control_df = full_control_df.merge(contrl_zcodecount, on = "PersonId")

In [45]:
full_control_df.rename(columns={'Obstetriccare': 'obstetric_care'}, inplace=True)

In [67]:
table1_cols = [
    # groups
    'weights',

    # demographics
    'age_at_delivery',
    'race_ethnicity',
    'Income',
    'parity',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',

    # outcomes
    'zcode_count',
    'gestational_week',
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

In [68]:
full_control_df = full_control_df[table1_cols].copy()
df_control_sub = df_control_sub[table1_cols].copy()
df_treated_sub = df_treated_sub[table1_cols].copy()
full_med_df = full_med_df[table1_cols].copy()

In [69]:
control_w = pd.concat([full_control_df, df_control_sub], ignore_index=True)
med_w = pd.concat([full_med_df, df_treated_sub], ignore_index=True)

In [70]:
categorical = [
    # groups
    'weights',

    # demographics
    'race_ethnicity',
    'Income',
    'parity',

    # comorbidities
    't2d_before_pregnancy',
    'hyper_before_pregnancy',
    'depression',
    'prior_Csection',
    'Prior_Preterm_Birth',
    'obstetric_care',

    # outcomes
    'gest_diabetes_no_prior_t2d',
    'preg_related_htn',
    'excessive_fetal_weight',
    'intra_grow_restrict',
    'csection',
    'preterm'
]

# 这两个变量按 median [Q1, Q3] / IQR 展示
nonnormal = [
    "gestational_week"
]

control_w['weights'] = control_w['weights'].astype(str)

# 可选但推荐：把 nonnormal 变量先转成 numeric
for col in nonnormal:
    if col in control_w.columns:
        control_w[col] = pd.to_numeric(control_w[col], errors='coerce')

table1 = TableOne(
    data=control_w,
    columns=table1_cols,
    categorical=categorical,
    groupby='weights',
    nonnormal=nonnormal,   # 关键改动
    pval=True
)

#table1.to_csv(data_dir / "controlw_table1_tableone.csv")
table1

In [71]:
from pathlib import Path
med_w['weights'] = med_w['weights'].astype(str)
data_dir = Path(output_path_local) / "revise_results"
for col in nonnormal:
    if col in med_w.columns:
        med_w[col] = pd.to_numeric(med_w[col], errors='coerce')

table1 = TableOne(
    data=med_w,
    columns=table1_cols,
    categorical=categorical,
    groupby='weights',
    nonnormal=nonnormal,   # 关键改动
    pval=True
)

table1.to_csv(data_dir / "med_w_table1_tableone.csv")
table1

### post hoc for GWG_category

In [55]:
from pathlib import Path
data_dir = Path(output_path_local) / "revise_results"

files = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non": "continued_vs_non_matched.csv",
    "former_vs_non": "former_vs_non_matched.csv"
}

In [57]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests


# -----------------------------
# Your file setup
# -----------------------------
data_dir = Path(output_path_local) / "revise_results"

files = {
    "continued_vs_former": "continued_vs_former_matched.csv",
    "continued_vs_non": "continued_vs_non_matched.csv",
    "former_vs_non": "former_vs_non_matched.csv"
}


# -----------------------------
# Main analysis function
# -----------------------------
def analyze_gwg_table(
    df: pd.DataFrame,
    group_col: str = "user_group",
    gwg_col: str = "gwg_category",
    alpha: float = 0.05
):
    """
    Analyze a 2 x K contingency table for user_group vs gwg_category.

    Returns:
        dict with:
            contingency_table
            expected_counts
            chi2_results
            standardized_residuals
            category_posthoc
    """
    # Keep only rows with both variables present
    dat = df[[group_col, gwg_col]].dropna().copy()

    # Force string for cleaner labels
    dat[group_col] = dat[group_col].astype(str)
    dat[gwg_col] = dat[gwg_col].astype(str)

    # Contingency table: rows = groups, cols = GWG categories
    ct = pd.crosstab(dat[group_col], dat[gwg_col], dropna=False)

    # Safety check: this workflow is for 2-group comparisons
    if ct.shape[0] != 2:
        raise ValueError(
            f"Expected exactly 2 groups in '{group_col}', but found {ct.shape[0]}: {list(ct.index)}"
        )

    # Overall chi-square test
    chi2, p_value, dof, expected = chi2_contingency(ct)

    expected_df = pd.DataFrame(expected, index=ct.index, columns=ct.columns)

    # Standardized residuals for cell-level interpretation
    # statsmodels Table handles general two-way contingency tables
    table_obj = Table(ct.values)
    std_resid = pd.DataFrame(
        table_obj.standardized_resids,
        index=ct.index,
        columns=ct.columns
    )

    # Category-wise 2-group proportion tests
    # For each GWG category: compare proportion in group 1 vs group 2
    group_names = list(ct.index)
    g1, g2 = group_names[0], group_names[1]
    n1 = ct.loc[g1].sum()
    n2 = ct.loc[g2].sum()

    posthoc_rows = []

    for category in ct.columns:
        count1 = int(ct.loc[g1, category])
        count2 = int(ct.loc[g2, category])

        counts = np.array([count1, count2])
        nobs = np.array([n1, n2])

        z_stat, p_raw = proportions_ztest(count=counts, nobs=nobs, alternative="two-sided")

        prop1 = count1 / n1 if n1 > 0 else np.nan
        prop2 = count2 / n2 if n2 > 0 else np.nan

        posthoc_rows.append({
            "gwg_category": category,
            "group_1": g1,
            "count_1": count1,
            "n_1": n1,
            "prop_1": prop1,
            "group_2": g2,
            "count_2": count2,
            "n_2": n2,
            "prop_2": prop2,
            "risk_difference": prop1 - prop2,
            "z_stat": z_stat,
            "p_raw": p_raw
        })

    posthoc_df = pd.DataFrame(posthoc_rows)

    # Holm adjustment across the GWG categories
    reject, p_holm, _, _ = multipletests(posthoc_df["p_raw"], alpha=alpha, method="holm")
    posthoc_df["p_holm"] = p_holm
    posthoc_df["significant_holm"] = reject

    # Helpful flags for residuals
    resid_flags = std_resid.copy()
    resid_flags = resid_flags.applymap(
        lambda x: (
            ">|2.58|" if abs(x) > 2.58
            else ">|1.96|" if abs(x) > 1.96
            else ""
        )
    )

    chi2_results = {
        "chi2": chi2,
        "dof": dof,
        "p_value": p_value,
        "n_total": int(ct.values.sum()),
        "min_expected": float(expected_df.min().min()),
        "cells_expected_lt_5": int((expected_df < 5).sum().sum()),
        "significant": bool(p_value < alpha),
    }

    return {
        "contingency_table": ct,
        "expected_counts": expected_df,
        "chi2_results": chi2_results,
        "standardized_residuals": std_resid,
        "residual_flags": resid_flags,
        "category_posthoc": posthoc_df,
    }


# -----------------------------
# Run all three datasets
# -----------------------------
all_results = {}

for comparison_name, filename in files.items():
    path = data_dir / filename
    df = pd.read_csv(path)

    result = analyze_gwg_table(
        df=df,
        group_col="user_group",
        gwg_col="gwg_category",
        alpha=0.05
    )

    all_results[comparison_name] = result

    print("\n" + "=" * 80)
    print(f"Comparison: {comparison_name}")
    print("=" * 80)

    print("\nContingency table")
    print(result["contingency_table"])

    print("\nExpected counts")
    print(result["expected_counts"].round(2))

    print("\nOverall chi-square")
    for k, v in result["chi2_results"].items():
        print(f"{k}: {v}")

    print("\nStandardized residuals")
    print(result["standardized_residuals"].round(3))

    print("\nResidual flags")
    print(result["residual_flags"])

    print("\nCategory-specific post hoc tests (Holm-adjusted)")
    print(
        result["category_posthoc"][
            [
                "gwg_category",
                "group_1", "count_1", "n_1", "prop_1",
                "group_2", "count_2", "n_2", "prop_2",
                "risk_difference", "z_stat", "p_raw", "p_holm", "significant_holm"
            ]
        ].round(4)
    )

In [58]:
summary_rows = []

for comparison_name, result in all_results.items():
    chi = result["chi2_results"]
    summary_rows.append({
        "comparison": comparison_name,
        "n_total": chi["n_total"],
        "chi2": chi["chi2"],
        "dof": chi["dof"],
        "p_value": chi["p_value"],
        "min_expected": chi["min_expected"],
        "cells_expected_lt_5": chi["cells_expected_lt_5"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.round(4)

In [60]:
from pathlib import Path
import pandas as pd

output_dir = Path(output_path_local) / "revise_results" / "csv_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

for comparison_name, result in all_results.items():

    # ---- 1. Contingency table (with %)
    ct = result["contingency_table"].copy()
    row_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_formatted = ct.astype(str) + " (" + row_pct.round(1).astype(str) + "%)"

    ct_formatted = ct_formatted.reset_index()

    # ---- 2. Expected counts
    expected = result["expected_counts"].round(2).reset_index()

    # ---- 3. Chi-square
    chi_df = pd.DataFrame([result["chi2_results"]])

    # ---- 4. Post hoc
    posthoc = result["category_posthoc"].copy()

    # 转换为百分比
    for col in ["prop_1", "prop_2", "risk_difference"]:
        if col in posthoc.columns:
            posthoc[col] = (posthoc[col] * 100).round(1)

    # ---- Save separately
    ct_formatted.to_csv(output_dir / f"{comparison_name}_table3.csv", index=False)
    expected.to_csv(output_dir / f"{comparison_name}_expected.csv", index=False)
    chi_df.to_csv(output_dir / f"{comparison_name}_chi2.csv", index=False)
    posthoc.to_csv(output_dir / f"{comparison_name}_posthoc.csv", index=False)

print(f"All CSV files saved to: {output_dir}")